<a href="https://colab.research.google.com/github/muhallilahnaf/RAGDSS/blob/master/RAGDSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## in terminal
## apt-get install zstd
## curl -fsSL https://ollama.com/install.sh | sh
## ollama serve &
## ollama run llama3.2, sqlcode:7b

In [5]:
!pip install ollama
!pip install chromadb
!pip install sqlglot
!pip install openrouter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 8.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentele

In [3]:
!wget https://raw.githubusercontent.com/muhallilahnaf/RAGDSS/refs/heads/master/prices_clean.csv
!wget https://raw.githubusercontent.com/muhallilahnaf/RAGDSS/refs/heads/master/reviews_clean.csv

--2026-04-27 17:37:57--  https://raw.githubusercontent.com/muhallilahnaf/RAGDSS/refs/heads/master/prices_clean.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 44971 (44K) [text/plain]
Saving to: ‘prices_clean.csv’

prices_clean.csv    100%[===================>]  43.92K  --.-KB/s    in 0.007s  

2026-04-27 17:37:57 (6.04 MB/s) - ‘prices_clean.csv’ saved [44971/44971]

--2026-04-27 17:37:57--  https://raw.githubusercontent.com/muhallilahnaf/RAGDSS/refs/heads/master/reviews_clean.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length

In [6]:
import sqlite3
import uuid
import pandas as pd
import ollama
import json
import chromadb
import sqlglot
from sqlglot.optimizer.qualify import qualify
from openrouter import OpenRouter

from google.colab import userdata
userdata.get('OPENROUTER')

In [15]:
with OpenRouter(api_key='OPENROUTER') as client:
    response = client.chat.send(
        model="openai/gpt-oss-20b:free",
        messages=[
            {"role": "user", "content": "What is the meaning of life?"}
        ],
    )

    print(response.choices[0].message.content)


The “meaning of life” is a question that has occupied philosophers, theologians, scientists, artists, and ordinary people for millennia, and there is no single answer that satisfies everyone.  Instead, most people find it helpful to think of meaning in a few interrelated ways:

| Perspective | What it emphasizes | How it can shape a life |
|-------------|--------------------|------------------------|
| **Existentialist** | Freedom, responsibility, authenticity | Make choices that reflect your own values, even when that feels risky or lonely. |
| **Religious/Spiritual** | Divine purpose, moral order, transcendence | Seek connection with a higher power or community, find guidance in sacred texts or practices. |
| **Humanistic Psychology** | Self‑actualization, belonging, personal growth | Cultivate relationships, pursue mastery and contribution, pursue goals that feel intrinsically rewarding. |
| **Narrative** | Constructing a coherent story | Reframe past experiences, set goals, see you

In [30]:
# test to check if llm is responding
# response = ollama.chat(model='llama3.2', messages=[
#   {'role': 'user', 'content': 'Why is the sky blue?'},
# ])
# print(response['message']['content'])

In [16]:
# start a database connection
conn = sqlite3.connect('RAGDSS.db')
cursor = conn.cursor()

# read prices csv
df = pd.read_csv('prices_clean.csv')
df.head()

,id,brand,categories,dateAdded,dateUpdated,name,currency,amount,date,isSale,merchant,shipping,condition
0,AV000tWuGV-KLJ3ac2-b,Amazon,Amazon Devices,2017-07-12T03:23:38Z,2017-07-25T02:04:06Z,Echo Show - Black,USD,229.99,2017-07-21T23:11:11.925Z,False,Amazon.com,FREE Shipping.,NaN
1,AV00l7jV-jtxr-f30lnX,Amazon,Amazon Devices,2017-07-12T02:19:04Z,2017-08-13T08:29:10Z,Amazon Echo Dot Case (fits Echo Dot 2nd Genera...,USD,19.99,2017-07-26T00:00:12.394Z,False,Amazon.com,FREE Shipping on orders over USD 25.00,NaN
2,AV00lzP7GV-KLJ3ac0uk,Amazon,Amazon Devices,2017-07-12T02:18:30Z,2017-08-13T08:28:52Z,Amazon Echo Dot Case (fits Echo Dot 2nd Genera...,USD,14.99,2017-07-25T23:52:21.319Z,False,Amazon.com,FREE Shipping on orders over USD 25.00,NaN
3,AV00lzP7GV-KLJ3ac0uk,Amazon,Amazon Devices,2017-07-12T02:18:30Z,2017-08-13T08:28:52Z,Amazon Echo Dot Case (fits Echo Dot 2nd Genera...,USD,9.99,2017-07-25T23:52:21.319Z,True,Amazon.com,FREE Shipping on orders over USD 25.00,NaN
4,AV00lzd5GV-KLJ3ac0ul,Amazon,Amazon Devices,2017-07-12T02:18:31Z,2017-07-18T23:46:14Z,All-New Amazon Fire TV Game Controller,USD,49.99,2017-06-30T15:54:25.350Z,False,easy-to-open packaging,FREE Shipping.,NaN


In [17]:
# add prices data to database
df.to_sql('prices', conn)

189

In [18]:
# test to check database works
cursor.execute("SELECT * FROM prices LIMIT 1")
print(cursor.fetchall())

[(0, 'AV000tWuGV-KLJ3ac2-b', 'Amazon', 'Amazon Devices', '2017-07-12T03:23:38Z', '2017-07-25T02:04:06Z', 'Echo Show - Black', 'USD', 229.99, '2017-07-21T23:11:11.925Z', 0, 'Amazon.com', 'FREE Shipping.', None)]


In [19]:
# get table description
cursor.execute("PRAGMA table_info(prices)")
columns = cursor.fetchall()
columns_info = [(col[1], col[2]) for col in columns]
table_description = "Columns:\n" + "\n".join([f"  - {name}: {col_type}" for name, col_type in columns_info])
print(table_description)

Columns:
  - index: INTEGER
  - id: TEXT
  - brand: TEXT
  - categories: TEXT
  - dateAdded: TEXT
  - dateUpdated: TEXT
  - name: TEXT
  - currency: TEXT
  - amount: REAL
  - date: TEXT
  - isSale: INTEGER
  - merchant: TEXT
  - shipping: TEXT
  - condition: TEXT


In [20]:
# generate sql using llm
table_name = 'prices'
context = """
    Your task is to produce correct SQL which will be used to perform SQL queries on a table. ONLY RETURN SQL STATEMENT.
    The table is named '{table_name}'. Its description is as follows:
    {table_description}
""".format(table_name=table_name, table_description=table_description)

with OpenRouter(api_key='OPENROUTER') as client:
  response = client.chat.send(
    model="openai/gpt-oss-20b:free",
    messages=[
      {"role": "user", "content": context + '\n\nQuery: find the id with the highest amount.'}
    ],
  )
  print(response.choices[0].message.content)

# response = ollama.chat(model='llama3.2', messages=[
#   {'role': 'user', 'content': context + '\n\nQuery: find the id with the highest amount.'},
# ])
# print(response['message']['content'])

SELECT id
FROM prices
ORDER BY amount DESC
LIMIT 1;


In [21]:
# validate sql
# llm_sql = response['message']['content']
llm_sql = response.choices[0].message.content
q = ''
for name, col_type in columns_info:
  q += f'"{name}": "{col_type}",'
# remove last comma
q = q[:-1]
schema = '{' + f'"{table_name}"' + ': {' + q + '}}'
print(schema)
schema = json.loads(schema)
expression = sqlglot.parse_one(llm_sql, read="sqlite")
qualified_expression = qualify(expression, schema=schema, dialect="sqlite")
print(qualified_expression.sql(dialect="sqlite"))

{"prices": {"index": "INTEGER","id": "TEXT","brand": "TEXT","categories": "TEXT","dateAdded": "TEXT","dateUpdated": "TEXT","name": "TEXT","currency": "TEXT","amount": "REAL","date": "TEXT","isSale": "INTEGER","merchant": "TEXT","shipping": "TEXT","condition": "TEXT"}}
SELECT "prices"."id" AS "id" FROM "prices" AS "prices" ORDER BY "prices"."amount" DESC LIMIT 1


In [22]:
# execute validated sql
cursor.execute(qualified_expression.sql(dialect="sqlite"))
rows = cursor.fetchall()
print(rows)

[('AVpfpzCi1cnluZ0-oxEr',)]


In [23]:
# read reviews data
dfr = pd.read_csv('reviews_clean.csv')
dfr.head()

,pid,reviews.date,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.text,reviews.title,reviews.username
0,AVpe7AsMilAPnD_xQ78G,2015-08-08T00:00:00.000Z,NaN,139.0,5.0,I initially had trouble deciding between the p...,"Paperwhite voyage, no regrets!",Cristina M
1,AVpe7AsMilAPnD_xQ78G,2015-09-01T00:00:00.000Z,NaN,126.0,5.0,Allow me to preface this with a little history...,One Simply Could Not Ask For More,Ricky
2,AVpe7AsMilAPnD_xQ78G,2015-07-20T00:00:00.000Z,NaN,69.0,4.0,I am enjoying it so far. Great for reading. Ha...,Great for those that just want an e-reader,Tedd Gardiner
3,AVpe7AsMilAPnD_xQ78G,2017-06-16T00:00:00.000Z,NaN,2.0,5.0,I bought one of the first Paperwhites and have...,Love / Hate relationship,Dougal
4,AVpe7AsMilAPnD_xQ78G,2016-08-11T00:00:00.000Z,NaN,17.0,5.0,I have to say upfront - I don't like coroporat...,I LOVE IT,Miljan David Tanic


In [24]:
# fetch rows with target product id
dfr_target = dfr[dfr['pid']==rows[0][0]]
dfr_target

,pid,reviews.date,reviews.doRecommend,reviews.numHelpful,reviews.rating,reviews.text,reviews.title,reviews.username
106,AVpfpzCi1cnluZ0-oxEr,NaN,NaN,402.0,3.0,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,B. Tarbuck
107,AVpfpzCi1cnluZ0-oxEr,2013-11-07T00:00:00.000Z,NaN,330.0,4.0,"Simply put, this is the Amazons iPad Air Kille...",An enlarged version of the 7 HDX - Same good i...,Amazon Reviewer
108,AVpfpzCi1cnluZ0-oxEr,2015-03-15T00:00:00.000Z,NaN,85.0,1.0,"After 15 months, my 500+ tablet is no longer u...",Dead after 15 months,Samuel Ruiz
109,AVpfpzCi1cnluZ0-oxEr,2014-10-30T00:00:00.000Z,NaN,59.0,3.0,Cons:1. Built on Android does not mean you can...,Not everything I expected...,Tina L. Harmon
110,AVpfpzCi1cnluZ0-oxEr,2013-11-07T00:00:00Z,NaN,NaN,NaN,"Simply put, this is the Amazons iPad Air Kille...",An enlarged version of the 7 HDX - Same good i...,Amazon Reviewer
111,AVpfpzCi1cnluZ0-oxEr,2013-11-07T00:00:00Z,NaN,NaN,NaN,"Simply put, this is the Amazons iPad Air Kille...",An enlarged version of the 7 HDX - Same good i...,Amazon Reviewer
112,AVpfpzCi1cnluZ0-oxEr,NaN,NaN,NaN,NaN,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,B. Tarbuck
113,AVpfpzCi1cnluZ0-oxEr,NaN,NaN,NaN,3.0,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,B. Tarbuck
114,AVpfpzCi1cnluZ0-oxEr,2013-11-07T00:00:00Z,NaN,NaN,NaN,"Simply put, this is the Amazons iPad Air Kille...",An enlarged version of the 7 HDX - Same good i...,Amazon Reviewer
115,AVpfpzCi1cnluZ0-oxEr,NaN,NaN,NaN,3.0,Updated 12/8/2014One year in...This review is ...,KF HDX 8.9 is ok do homework on Prime download...,B. Tarbuck


In [25]:
# vectorize review data
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="reviews")
collection.add(
    ids=[str(uuid.uuid4()) for _ in range(len(dfr_target))],
    documents=dfr_target['reviews.text'].tolist(),
    metadatas=dfr_target[[
        'reviews.date',
        'reviews.doRecommend',
        'reviews.numHelpful',
        'reviews.rating',
        'reviews.title',
        'reviews.username'
    ]].to_dict(orient='records')
)


results = collection.query(
    query_texts=["Give a summary of the reviews"],
    n_results=2
)
print(results)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 30.6MiB/s]


{'ids': [['c9255809-112c-4156-abf4-1cd4deca56a2', '247d324b-b995-4de3-943c-42f375551725']], 'embeddings': None, 'documents': [["Cons:1. Built on Android does not mean you can have Google Play Apps. This is an unfortunate inconvenience, as there are some Google Apps that are just better than their Amazon App counterparts, and a few that I can not find within the Amazon Apps. If you want a fully functional tablet, then buy a tablet and download the free kindle app. If you want to pay more for something that does less, then the kindle is for you. This was a gift, so I will keep it, and continue to use my phone for the rest of my Apps that are not available on the kindle.2. Camera is less than perfect. Limited settings available. Again, I will use my phone for pics, the same scene with my phone showed the flames, with a small amount of glare. (Husband is a firefighter, so I take a lot of pictures of their exercises for their training.) An old Casio Exilim C721 phone with 5MP camera still t

In [29]:
query = 'Question: Give a summary of the reviews. Context: ' + ' '.join(results['documents'][0])
with OpenRouter(api_key='OPENROUTER') as client:
  response = client.chat.send(
    model="openai/gpt-oss-20b:free",
    messages=[
      {"role": "user", "content": query}
    ],
  )

  print(response.choices[0].message.content)

**Overall Verdict**

The Kindle Fire HDX is a solid mid‑tier media device with a bright, high‑resolution display, great audio, a faster processor than its predecessor, and a decently long battery life. The build feels sturdy and the interface is intuitive for basic navigation. However, its appeal is largely confined to Amazon‑centric content, and several key shortcomings make it less than an all‑purpose tablet.

---

### What’s Working Well  

| Area | Highlights |
|------|------------|
| **Display** | 8.9‑inch Full HD (1920×1200) with excellent contrast and color accuracy. |
| **Sound** | Dual speakers provide clear audio; no “blue haze” issue reported. |
| **Performance** | A 2‑GHz dual‑core processor that feels snappy in everyday tasks. |
| **Build Quality** | Solid chassis, no visible gaskets or haze; more robust than the previous Fire HD. |
| **Battery** | Slightly longer endurance than the older model (≈4–5 hrs video). |
| **Interface** | Simple, touch‑friendly UI; settings large